# 06 · Gradients and derivatives: which way is downhill?

*Neural Networks from Scratch: Build, Train and Understand AI Models*, notebook 6 of 12

A loss tells you how wrong the network is. To improve it, you need to know how the loss **changes** when you change each weight: a little up, or a little down? That information is the **gradient**. In this notebook you measure derivatives numerically, follow the chain rule through a single neuron by hand, and build the tools that professionals use to check that a gradient is correct.

## What you will build

- `numerical_derivative(f, x, h)`: the slope of a function of one number, from two evaluations;
- `numerical_gradient(f, array, eps)`: the slope of a loss with respect to **every** entry of a weight array;
- `relative_error(a, b)`: a scale-free way to say whether two gradients agree.

## Learning objectives

By the end of this notebook you will be able to:

1. read a derivative as the slope of a function and a partial derivative as the slope along one input;
2. estimate derivatives with central differences, and choose the step size $h$ sensibly;
3. apply the chain rule through a computational graph, forwards and backwards, by hand;
4. explain what a gradient is, what shape it has, and why $-\nabla L$ points downhill;
5. gradient-check an analytic gradient against a numerical one with a relative error.

### How to work through this notebook

Run the cells from top to bottom. When you reach a **✍️ Your turn** cell, type the function yourself and then run the check cell beneath it:

- ✅ means your function matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not match yet. In both cases the notebook quietly uses a hidden reference version, so you can always keep going and come back later.

Each reference solution sits in a collapsed cell just above the function you type. Open it only when you are stuck.

## Dependencies

- **Tier 1:** NumPy for every calculation and Matplotlib for the plots. Nothing else is imported, so the notebook runs in JupyterLite in your browser, on the course JupyterHub server and in Google Colab.
- Nothing is downloaded, and every random number comes from a seeded generator.

## Setup

Run this cell first. It imports NumPy and Matplotlib, creates a seeded random generator and defines `practice`, the helper that checks each function you write against a reference.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import matplotlib.pyplot as plt

## 📦 Recap: what you built earlier

The hidden cell below re-defines the pieces from notebooks 01–05 under their usual names: `relu`, `relu_grad`, `sigmoid`, `softmax`, `dense_forward`, `init_network`, `forward`, `predict`, `accuracy`, `make_spirals`, and the losses `mse`, `binary_cross_entropy`, `cross_entropy` and `cross_entropy_from_logits`. Run it; open it only if you want to look back at the code.

In [ ]:
# 📦 Recap: the pieces from notebooks 01–04, under their usual names.

def relu(Z):
    """ReLU, element by element: max(0, Z)."""
    return np.maximum(0.0, Z)


def relu_grad(Z):
    """The slope of ReLU: 1.0 where Z > 0, else 0.0 (we take 0 at Z = 0)."""
    return (Z > 0).astype(float)


def sigmoid(Z):
    """Numerically stable logistic function 1 / (1 + exp(-Z)), element by element."""
    return np.exp(-np.logaddexp(0.0, -Z))


def softmax(Z):
    """Row-wise softmax of logits Z (batch, classes). Subtracting each row's maximum avoids overflow."""
    exps = np.exp(Z - Z.max(axis=1, keepdims=True))
    return exps / exps.sum(axis=1, keepdims=True)


def dense_forward(X, W, b):
    """Z = X @ W + b. X: (batch, n_in), W: (n_in, n_out), b: (n_out,) -> Z: (batch, n_out)."""
    return X @ W + b


def init_network(sizes, rng):
    """He-initialised layers for sizes such as [2, 64, 3]: a list of {"W": (n_in, n_out), "b": (n_out,)}."""
    return [
        {"W": rng.normal(0.0, np.sqrt(2.0 / n_in), (n_in, n_out)), "b": np.zeros(n_out)}
        for n_in, n_out in zip(sizes[:-1], sizes[1:])
    ]


def forward(network, X):
    """ReLU hidden layers and a softmax output.

    Returns (probs, cache): probs has shape (batch, classes), and cache[i] = (A_prev, Z) for layer i.
    """
    cache = []
    A = X
    for index, layer in enumerate(network):
        Z = dense_forward(A, layer["W"], layer["b"])
        cache.append((A, Z))
        A = softmax(Z) if index == len(network) - 1 else relu(Z)
    return A, cache


def predict(network, X):
    """The predicted class index for every row of X: shape (batch,)."""
    probs, _ = forward(network, X)
    return probs.argmax(axis=1)


def accuracy(network, X, y):
    """The fraction of rows of X whose predicted class equals the label in y."""
    return float(np.mean(predict(network, X) == y))


def make_spirals(n_per_class, n_classes, noise, rng):
    """n_classes interleaved spiral arms in 2-D, sorted by class.

    Returns X with shape (n_per_class * n_classes, 2) and integer labels y with
    shape (n_per_class * n_classes,). noise is the spread of the random angle jitter.
    """
    radius = np.linspace(0.05, 1.0, n_per_class)
    X_parts, y_parts = [], []
    for k in range(n_classes):
        angle = 2 * np.pi * k / n_classes + 6.0 * radius + rng.normal(0.0, noise, n_per_class)
        X_parts.append(np.column_stack([radius * np.cos(angle), radius * np.sin(angle)]))
        y_parts.append(np.full(n_per_class, k))
    return np.concatenate(X_parts), np.concatenate(y_parts)


# From notebook 05: loss functions.

def mse(pred, target):
    """Mean squared error over every entry of two arrays with the same shape."""
    return float(np.mean((pred - target) ** 2))


def binary_cross_entropy(p, y):
    """Mean binary cross-entropy of probabilities p (batch,) against 0/1 labels y (batch,), clipped."""
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def cross_entropy(probs, y):
    """Mean negative log-probability of the true class. probs: (batch, classes), y: (batch,) ints."""
    picked = probs[np.arange(len(y)), y]
    return float(-np.mean(np.log(np.clip(picked, 1e-12, None))))


def cross_entropy_from_logits(Z, y):
    """The same loss computed from logits Z (batch, classes) with the log-sum-exp trick."""
    row_max = Z.max(axis=1, keepdims=True)
    log_sum_exp = row_max[:, 0] + np.log(np.exp(Z - row_max).sum(axis=1))
    return float(np.mean(log_sum_exp - Z[np.arange(len(y)), y]))

## 1 · A derivative is a slope

Take a function of one number, such as $f(x) = x^3 - 2x$. Between two nearby points the function rises by $f(x + h) - f(x)$ while $x$ moves by $h$, so the slope of the straight line through both points (a **secant**) is

$$\frac{f(x + h) - f(x)}{h} .$$

As $h$ shrinks, the secant turns into the **tangent**, and its slope becomes the **derivative** $f'(x)$. A positive derivative means "increasing $x$ increases $f$"; a negative one means the opposite; and its size says how strongly $f$ reacts. For $f(x) = x^3 - 2x$ calculus gives $f'(x) = 3x^2 - 2$, so $f'(1.5) = 4.75$.

The cell below computes secant slopes at $x = 1.5$ for smaller and smaller $h$.

In [ ]:
def cubic(x):
    return x ** 3 - 2 * x

for h in [1.0, 0.1, 0.01, 0.001]:
    print(f"h = {h:<6} secant slope = {(cubic(1.5 + h) - cubic(1.5)) / h:.6f}   (the exact derivative is 4.75)")

In [ ]:
xs = np.linspace(-0.5, 2.6, 200)
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(xs, cubic(xs), color="#2a78d6", linewidth=2, label=r"$f(x) = x^3 - 2x$")
ax.plot(xs, cubic(1.5) + 4.75 * (xs - 1.5), color="#1baf7a", linewidth=2, label="tangent at x = 1.5 (slope 4.75)")
secant_slope = (cubic(2.5) - cubic(1.5)) / 1.0
ax.plot(xs, cubic(1.5) + secant_slope * (xs - 1.5), color="#eb6834", linewidth=2, linestyle="--",
        label=f"secant with h = 1 (slope {secant_slope:.2f})")
ax.plot([1.5, 2.5], [cubic(1.5), cubic(2.5)], "o", color="#eb6834", markersize=8)
ax.set_ylim(-3, 12)
ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.grid(alpha=0.3)
ax.legend()
ax.set_title("As h shrinks, the secant turns into the tangent")
plt.show()

The slopes approach $4.75$ only slowly: the error shrinks in proportion to $h$. Taylor's theorem explains why. Expanding around $x$,

$$f(x \pm h) = f(x) \pm h f'(x) + \tfrac{h^2}{2} f''(x) \pm \tfrac{h^3}{6} f'''(x) + \dots$$

so the one-sided (**forward**) difference is off by about $\tfrac{h}{2} f''(x)$. Subtracting the two expansions instead cancels the $h^2$ terms:

$$\frac{f(x + h) - f(x - h)}{2h} = f'(x) + \tfrac{h^2}{6} f'''(x) + \dots$$

This **central difference** looks equally far to both sides, and its error shrinks with $h^2$: ten times smaller $h$ gives a hundred times smaller error, for the same two evaluations of $f$.

### ✍️ Your turn: `numerical_derivative`

Write `numerical_derivative(f, x, h=1e-5)`:

- **inputs:** `f`, a function that takes one number and returns one number; `x`, the point; `h`, the step;
- **output:** the central-difference estimate $\dfrac{f(x + h) - f(x - h)}{2h}$.

Hint: it is one line. Use brackets around `2 * h`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def numerical_derivative_reference(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

In [ ]:
def numerical_derivative(f, x, h=1e-5):
    """Central-difference estimate of f'(x) for a function f of one number."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
numerical_derivative = practice("numerical_derivative", numerical_derivative, numerical_derivative_reference, cases=[
    (np.sin, 1.0, 1e-5),
    (cubic, 1.5, 1e-3),
    (np.exp, 0.0, 1e-4),
    (np.abs, 0.0, 1e-5),   # edge case: a kink, where the true derivative does not exist
])
print(f"central difference of the cubic at 1.5: {numerical_derivative(cubic, 1.5):.10f}")

**Expected behaviour.** With the default $h = 10^{-5}$ the cubic's slope at $1.5$ comes out as $4.7500000001$. The error is $10^{-10}$, exactly the $\tfrac{h^2}{6} f'''(x) = h^2$ that the Taylor expansion predicts for this cubic, whereas the forward difference with $h = 0.001$ was only correct to two decimal places. At the kink of $|x|$ the central difference returns $0$, the average of the slopes $-1$ and $+1$ on either side, even though no derivative exists there. Keep that in mind: ReLU has the same kind of kink at $0$.

## 2 · 🧪 How small should the step be?

If the error shrinks with $h^2$, why not use $h = 10^{-15}$? Because computers round. A `float64` keeps about 16 significant digits, so each value of $f$ carries a rounding error of roughly $\varepsilon \approx 2.2 \times 10^{-16}$ times its size. The difference quotient divides that error by $h$. Two errors therefore compete:

- the **truncation error** of the formula, which shrinks as $h$ shrinks: about $h$ for forward differences and $h^2$ for central ones;
- the **rounding error**, about $\varepsilon / h$, which *grows* as $h$ shrinks.

The total is smallest where the two balance: near $h \approx \sqrt{\varepsilon} \approx 10^{-8}$ for forward differences and $h \approx \varepsilon^{1/3} \approx 10^{-5}$ for central ones. That is why `1e-5` is the default step in this course. Run the sweep below, then change `x0` and run it again.

In [ ]:
x0 = 1.0             # try 0.1 and 30.0, then run the cell again
exact = np.cos(x0)   # the true derivative of sin at x0
steps = 10.0 ** -np.arange(1, 16)
forward_errors = np.array([abs((np.sin(x0 + h) - np.sin(x0)) / h - exact) for h in steps])
central_errors = np.array([abs(numerical_derivative(np.sin, x0, h) - exact) for h in steps])

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.loglog(steps, np.maximum(forward_errors, 1e-17), "o-", color="#eb6834", linewidth=2, label="forward difference")
ax.loglog(steps, np.maximum(central_errors, 1e-17), "o-", color="#2a78d6", linewidth=2, label="central difference")
ax.invert_xaxis()
ax.set_xlabel("step size h (smaller to the right)")
ax.set_ylabel("absolute error")
ax.grid(alpha=0.3)
ax.legend()
ax.set_title("Truncation error falls, then rounding error takes over")
plt.show()
print(f"best forward step: {steps[forward_errors.argmin()]:.0e} (error {forward_errors.min():.1e})")
print(f"best central step: {steps[central_errors.argmin()]:.0e} (error {central_errors.min():.1e})")

**What to notice.** Reading the plot from left to right, both errors fall at first: the forward error by a factor of about 10 for each tenfold smaller step and the central error by about 100. Then the rounding error takes over and both curves climb again. At $h = 10^{-15}$ both estimates are worse than at $h = 0.1$. For `x0 = 1.0` the best central step is near $10^{-5}$, with an error around $10^{-11}$, and the best forward step is near $10^{-8}$, with an error around $10^{-8}$.

## 3 · Partial derivatives and the gradient

A loss depends on many numbers at once. For a function of several inputs, $f(w_0, w_1, \dots)$, the **partial derivative** $\partial f / \partial w_j$ is the slope along one input while all the others are held fixed. You estimate it exactly as before, nudging only $w_j$.

Collect all the partial derivatives into one array and you have the **gradient**:

$$\nabla f = \Big( \frac{\partial f}{\partial w_0},\ \frac{\partial f}{\partial w_1},\ \dots \Big).$$

The gradient has **the same shape as the thing you differentiate with respect to**: the gradient of a loss with respect to a weight matrix of shape `(n_in, n_out)` is another `(n_in, n_out)` array. It points in the direction in which $f$ increases fastest, so $-\nabla f$ points **downhill**, which is the direction training will follow.

For example, $f(w) = w_0^2 + 3 w_0 w_1 + \sin w_1$ has $\partial f / \partial w_0 = 2 w_0 + 3 w_1$ and $\partial f / \partial w_1 = 3 w_0 + \cos w_1$. At $w = (1, 2)$ these are $8$ and $3 + \cos 2 \approx 2.5839$. The cell below estimates the first one by nudging $w_0$ only.

In [ ]:
def two_variable_example(w):
    return float(w[0] ** 2 + 3 * w[0] * w[1] + np.sin(w[1]))

w_point = np.array([1.0, 2.0])
nudge = np.array([1e-5, 0.0])          # move w0 only; w1 stays fixed
slope_w0 = (two_variable_example(w_point + nudge) - two_variable_example(w_point - nudge)) / 2e-5
print(f"partial derivative with respect to w0: {slope_w0:.6f} (exact: 8)")

### ✍️ Your turn: `numerical_gradient`

Write `numerical_gradient(f, array, eps=1e-5)`, which estimates the partial derivative of `f` with respect to **every** entry of `array`:

- **inputs:** `f`, a function that takes the array and returns one number (a float); `array`, a float array of any shape, such as a weight matrix; `eps`, the step;
- **output:** an array of the same shape as `array` whose entry at each index is $\dfrac{f(\text{array with that entry} + \varepsilon) - f(\text{array with that entry} - \varepsilon)}{2\varepsilon}$.

Nudge the entries **in place**, one at a time, and put each one back exactly as it was before moving on. Working in place means `f` can also read the array some other way, for example through a network that holds it, which is exactly how you will check a network's gradients in notebook 07.

Hints:

- start with `grad = np.zeros(array.shape)`;
- `for index in np.ndindex(array.shape):` visits every index of an array of any shape;
- inside the loop, save `original = array[index]`, set `array[index] = original + eps` and evaluate `f(array)`, do the same with `original - eps`, then restore with `array[index] = original`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below.

In [ ]:
def numerical_gradient_reference(f, array, eps=1e-5):
    grad = np.zeros(array.shape)
    for index in np.ndindex(array.shape):
        original = array[index]
        array[index] = original + eps
        loss_plus = f(array)
        array[index] = original - eps
        loss_minus = f(array)
        array[index] = original
        grad[index] = (loss_plus - loss_minus) / (2 * eps)
    return grad

In [ ]:
def numerical_gradient(f, array, eps=1e-5):
    """Central-difference gradient of the number f(array) with respect to every entry of array.

    array: float array of any shape (nudged in place and restored). Returns an array shaped like array.
    """
    # ✍️ Your code here (about nine lines)
    raise NotImplementedError

In [ ]:
def sum_of_squares(a):
    return float(np.sum(a ** 2))

weights_in_a_closure = np.array([[0.5, -1.0], [2.0, 0.3]])

def loss_reading_a_closure(_):
    # Ignores its argument and reads the weights directly, the way a network's loss does.
    return float(np.sum(np.tanh(np.array([1.0, -2.0]) @ weights_in_a_closure)))

numerical_gradient = practice("numerical_gradient", numerical_gradient, numerical_gradient_reference, cases=[
    (sum_of_squares, np.array([[1.0, -2.0, 0.5], [3.0, 0.0, -1.0]])),
    (two_variable_example, np.array([1.0, 2.0])),
    (loss_reading_a_closure, weights_in_a_closure),
])

probe = np.array([[0.5, -1.5], [2.0, 3.0]])
probe_before = probe.copy()
numerical_gradient(sum_of_squares, probe)
if np.array_equal(probe, probe_before):
    print("✅ numerical_gradient leaves its array exactly as it found it.")
else:
    print("❌ numerical_gradient changed the array it was given: put each entry back after nudging it.")
    print("   The rest of the notebook uses the reference numerical_gradient.")
    numerical_gradient = numerical_gradient_reference

**Expected behaviour.** The gradient of the sum of squares is $2 \times$ the array, so `[[2, -4, 1], [6, 0, -2]]` for the first case, and the second case gives about `[8, 2.5839]`, matching the partial derivatives above. The result always has the shape of the array, and the array is unchanged afterwards. Notice the cost: two evaluations of `f` per entry. For a network with a million weights that is two million forward passes for one gradient, which is why notebook 07 introduces a much faster method and keeps this one for checking.

## 4 · The chain rule

Neural networks are built by composing simple functions, so we need the derivative of a composition. If $y = f(g(x))$, a small change in $x$ changes $g$ by about $g'(x)$ times as much, and that change in $g$ changes $f$ by $f'(g(x))$ times as much again:

$$\frac{dy}{dx} = f'(g(x)) \cdot g'(x) .$$

In words: **multiply the local slopes along the path.** For $y = \sin(x^2)$ the outer slope is $\cos(x^2)$, the inner slope is $2x$, and so $dy/dx = 2x \cos(x^2)$. Check it numerically below.

In [ ]:
def sin_of_square(x):
    return np.sin(x ** 2)

x_chain = 1.5
print(f"chain rule:           {2 * x_chain * np.cos(x_chain ** 2):.10f}")
print(f"numerical derivative: {numerical_derivative(sin_of_square, x_chain):.10f}")

## 5 · A computational graph for one neuron

Here is one sigmoid neuron with two inputs, its prediction and its loss, written as a chain of small steps. With the input $x = (1, 2)$, the weights $w = (0.5, -0.25)$, the bias $b = 0.1$ and the label $y = 1$:

```
 x ──┐
     ├─► [ u = w · x ] ──► [ z = u + b ] ──► [ p = σ(z) ] ──► [ L = −log p ]     (label y = 1)
 w ──┘                          ▲
                                b
```

**Forward pass** (left to right): $u = 0.5 \cdot 1 + (-0.25) \cdot 2 = 0$, then $z = 0.1$, then $p = \sigma(0.1) \approx 0.524979$, and finally $L = -\log p \approx 0.644397$.

**Backward pass** (right to left). Start from $\partial L / \partial L = 1$. At each box, multiply the gradient arriving from the right (the **upstream** gradient) by the box's **local** derivative:

| Step | Local derivative | Gradient of $L$ |
|---|---|---|
| $L = -\log p$ | $\partial L / \partial p = -1/p$ | $-1.904837$ |
| $p = \sigma(z)$ | $\partial p / \partial z = p(1 - p) \approx 0.249376$ | $\partial L / \partial z = -1.904837 \times 0.249376 \approx -0.475021$ |
| $z = u + b$ | $\partial z / \partial b = 1$, $\partial z / \partial u = 1$ | $\partial L / \partial b = \partial L / \partial u \approx -0.475021$ |
| $u = w \cdot x$ | $\partial u / \partial w_j = x_j$ | $\partial L / \partial w = -0.475021 \times (1, 2) \approx (-0.475021, -0.950042)$ |

Notice that $\partial L / \partial z = p - 1 = p - y$. That is no accident: for a sigmoid followed by binary cross-entropy, $\left(-\frac{y}{p} + \frac{1 - y}{1 - p}\right) \cdot p(1 - p) = p - y$ for either label. Over a batch of $n$ examples with the mean loss, each example contributes $(p_i - y_i)/n$, so

$$\frac{\partial L}{\partial w} = \frac{1}{n} X^\top (p - y), \qquad \frac{\partial L}{\partial b} = \frac{1}{n} \sum_i (p_i - y_i).$$

The next cell repeats the forward and backward passes in code and checks the result with your `numerical_gradient`.

In [ ]:
x_one = np.array([1.0, 2.0])
w_one = np.array([0.5, -0.25])
b_one = np.array([0.1])          # an array of shape (1,), so numerical_gradient can nudge it
label = 1.0

# Forward pass, one small step at a time
u = w_one @ x_one
z = u + b_one[0]
p = float(sigmoid(z))
loss = -np.log(p)
print(f"forward:  u = {u:.6f}   z = {z:.6f}   p = {p:.6f}   L = {loss:.6f}")

# Backward pass: upstream gradient times local derivative
dL_dp = -1.0 / p
dL_dz = dL_dp * p * (1 - p)
dL_db = dL_dz * 1.0
dL_dw = dL_dz * x_one
print(f"backward: dL/dp = {dL_dp:.6f}   dL/dz = {dL_dz:.6f} (p - y = {p - label:.6f})")
print(f"          dL/db = {dL_db:.6f}   dL/dw = {np.round(dL_dw, 6)}")

def one_neuron_loss(w, b):
    return binary_cross_entropy(sigmoid(np.array([x_one @ w + b[0]])), np.array([label]))

print("numerical dL/dw:", np.round(numerical_gradient(lambda w: one_neuron_loss(w, b_one), w_one), 6))
print("numerical dL/db:", np.round(numerical_gradient(lambda b: one_neuron_loss(w_one, b), b_one), 6))

The hand calculation and the numerical gradient agree to six decimal places. But "they look the same" is not a test you can automate. You need a single number that says how far apart two gradients are, whatever their size.

## 6 · Comparing gradients: the relative error

An absolute difference of $10^{-6}$ is excellent agreement for gradients of size $1$, but meaningless for gradients of size $10^{-8}$. The **relative error** divides the size of the difference by the sizes of the two gradients:

$$\text{relative error}(a, b) = \frac{\lVert a - b \rVert}{\max\big(\lVert a \rVert + \lVert b \rVert,\ 10^{-12}\big)} ,$$

where $\lVert \cdot \rVert$ is the Euclidean length of all entries together (`np.linalg.norm`). It is $0$ when the arrays are equal and at most $1$, which happens when they point in opposite directions. The tiny floor of $10^{-12}$ avoids dividing by zero when both arrays are zero.

As a rule of thumb with `float64` and `eps = 1e-5`: below about $10^{-7}$ the gradients agree; above about $10^{-4}$ there is almost certainly a bug.

In [ ]:
a = np.array([1.0, 2.0, 3.0])
print("norm of a:          ", np.linalg.norm(a))
print("norm of a - (a + 1e-6):", np.linalg.norm(a - (a + 1e-6)))

### ✍️ Your turn: `relative_error`

Write `relative_error(a, b)`:

- **inputs:** two arrays (or numbers) of the same shape;
- **output:** $\lVert a - b \rVert / \max(\lVert a \rVert + \lVert b \rVert, 10^{-12})$ as a Python `float`.

Hints: convert the inputs with `np.asarray(a, dtype=float)` so that plain numbers and lists work too; `np.linalg.norm(x)` gives the length of all entries of `x` together; Python's built-in `max` takes the larger of two numbers.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below.

In [ ]:
def relative_error_reference(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

In [ ]:
def relative_error(a, b):
    """||a - b|| / max(||a|| + ||b||, 1e-12) for two arrays of the same shape. Returns a float."""
    # ✍️ Your code here (two lines)
    raise NotImplementedError

In [ ]:
relative_error = practice("relative_error", relative_error, relative_error_reference, cases=[
    (np.array([1.0, 2.0, 3.0]), np.array([1.0, 2.0, 3.000001])),
    (np.array([[1.0, -1.0], [0.5, 2.0]]), -np.array([[1.0, -1.0], [0.5, 2.0]])),   # opposite directions: 1
    (np.zeros(3), np.zeros(3)),                                                    # edge case: both zero
])

**Expected behaviour.** Nearly identical gradients give a tiny number (about $1.3 \times 10^{-7}$ for the first case), opposite gradients give exactly $1$, and two zero arrays give $0$ rather than a division-by-zero error.

## 7 · Gradient-checking a sigmoid neuron on a batch

Now combine everything. A sigmoid neuron with three inputs makes predictions for a batch of eight examples, and its loss is the mean binary cross-entropy. Section 5 derived the **analytic** gradient, $\partial L / \partial w = X^\top (p - y) / n$ and $\partial L / \partial b = \sum_i (p_i - y_i) / n$. **Gradient checking** compares it with the numerical gradient, which uses nothing but evaluations of the loss. If the relative error is tiny, the formula is right.

**Shapes.** `X_neuron` is `(8, 3)`, `y_neuron` and `p` are `(8,)`, `w_neuron` is `(3,)` and `b_neuron` is `(1,)`. The gradients have the shapes of `w_neuron` and `b_neuron`.

In [ ]:
neuron_rng = np.random.default_rng(6)
X_neuron = neuron_rng.normal(size=(8, 3))
y_neuron = (neuron_rng.random(8) < 0.5).astype(float)
w_neuron = neuron_rng.normal(size=3)
b_neuron = np.array([0.1])

def neuron_loss(w, b):
    return binary_cross_entropy(sigmoid(X_neuron @ w + b), y_neuron)

def neuron_gradients(w, b):
    # Analytic gradients from section 5: dL/dz = (p - y) / n, then the chain rule.
    dz = (sigmoid(X_neuron @ w + b) - y_neuron) / len(y_neuron)
    return X_neuron.T @ dz, np.array([dz.sum()])

dw, db = neuron_gradients(w_neuron, b_neuron)
dw_numerical = numerical_gradient(lambda w: neuron_loss(w, b_neuron), w_neuron)
db_numerical = numerical_gradient(lambda b: neuron_loss(w_neuron, b), b_neuron)
print("analytic  dL/dw:", np.round(dw, 8), "  dL/db:", np.round(db, 8))
print("numerical dL/dw:", np.round(dw_numerical, 8), "  dL/db:", np.round(db_numerical, 8))
print(f"relative errors: w {relative_error(dw, dw_numerical):.1e}, b {relative_error(db, db_numerical):.1e}")

# A typical bug: forgetting to divide by the batch size.
dw_buggy = X_neuron.T @ (sigmoid(X_neuron @ w_neuron + b_neuron) - y_neuron)
print(f"relative error of the buggy gradient: {relative_error(dw_buggy, dw_numerical):.2f}")

**What to notice.** The correct gradient agrees with the numerical one to a relative error around $10^{-11}$. The buggy gradient, which forgot the division by $n = 8$, is too large by a factor of 8 and scores $7/9 \approx 0.78$. There is no grey area between those two numbers, and that is what makes gradient checking such a reliable test.

### 🧪 Try it: the step size in a gradient check

Gradient checks use `eps = 1e-5` for the same reason as section 2. Change `eps` and run the cell again to see the relative error of the **correct** gradient rise when the step is too large or too small.

In [ ]:
eps = 1e-5      # try 1e-1, 1e-8 and 1e-12, then run the cell again
dw_check = numerical_gradient(lambda w: neuron_loss(w, b_neuron), w_neuron, eps=eps)
print(f"eps = {eps:.0e}: relative error of the correct analytic gradient = {relative_error(dw, dw_check):.1e}")

With `eps = 1e-1` the truncation error pushes the relative error up to about $3 \times 10^{-4}$, and with `eps = 1e-12` rounding pushes it to nearly $10^{-4}$. Either could make a correct gradient look suspicious, or hide a small bug. The default `1e-5` sits at the bottom of the valley, at about $5 \times 10^{-12}$.

## 8 · Put it together: a gradient check you can trust

The cell below checks the neuron's analytic gradient at five random settings of the weights, confirms that the check rejects the buggy gradient, and then uses the gradient for what it is for: a small step **against** it must lower the loss.

In [ ]:
together_rng = np.random.default_rng(2024)
worst = 0.0
for trial in range(5):
    w_trial = together_rng.normal(size=3)
    b_trial = together_rng.normal(size=1)
    dw_trial, db_trial = neuron_gradients(w_trial, b_trial)
    dw_num = numerical_gradient(lambda w: neuron_loss(w, b_trial), w_trial)
    db_num = numerical_gradient(lambda b: neuron_loss(w_trial, b), b_trial)
    worst = max(worst, relative_error(dw_trial, dw_num), relative_error(db_trial, db_num))
print(f"worst relative error over 5 random settings: {worst:.1e}")

loss_before = neuron_loss(w_neuron, b_neuron)
step = 0.5
loss_after = neuron_loss(w_neuron - step * dw, b_neuron - step * db)
print(f"loss before a step against the gradient: {loss_before:.4f}, after: {loss_after:.4f}")

assert worst < 1e-7, "the analytic gradient should match the numerical one"
assert relative_error(dw_buggy, dw_numerical) > 0.1, "the check should reject a gradient that forgot the 1/n"
assert loss_after < loss_before, "a small step against the gradient should lower the loss"
assert abs(numerical_derivative(np.sin, 1.0) - np.cos(1.0)) < 1e-9
print("✅ All checks passed.")

**What to notice.** Every check agrees to better than $10^{-7}$, and stepping against the gradient lowers the loss. That one step is the whole idea of training. Notebook 08 simply repeats it thousands of times, using the fast gradients from notebook 07.

## Exercises

### Exercise 1 · The forward difference, measured

Write `forward_difference(f, x, h)`, the one-sided estimate $\big(f(x + h) - f(x)\big)/h$. Then answer the open question printed by the check cell: when $h$ shrinks by a factor of 10, by roughly what factor does each error shrink, and why?

In [ ]:
def forward_difference_reference(f, x, h):
    return (f(x + h) - f(x)) / h

In [ ]:
def forward_difference(f, x, h):
    """One-sided estimate (f(x + h) - f(x)) / h of f'(x)."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
forward_difference = practice("forward_difference", forward_difference, forward_difference_reference, cases=[
    (np.sin, 1.0, 1e-3),
    (cubic, 1.5, 0.1),
])
for h in [1e-2, 1e-3, 1e-4]:
    print(f"h = {h:.0e}: forward error {abs(forward_difference(np.sin, 1.0, h) - np.cos(1.0)):.1e}, "
          f"central error {abs(numerical_derivative(np.sin, 1.0, h) - np.cos(1.0)):.1e}")

### Exercise 2 · The gradient of a linear neuron with squared error

A linear neuron predicts $\hat{y} = Xw + b$ and is trained with $L = \frac{1}{n}\sum_i (\hat{y}_i - y_i)^2$. Use the chain rule to derive $\partial L / \partial w$ and $\partial L / \partial b$, then write `linear_neuron_mse_gradients(X, y, w, b)`, returning `(dw, db)` with the shapes of `w` `(features,)` and `b` `(1,)`. The check cell also gradient-checks your formula with `numerical_gradient`.

In [ ]:
def linear_neuron_mse_gradients_reference(X, y, w, b):
    residual = X @ w + b - y
    return 2 * X.T @ residual / len(y), np.array([2 * residual.sum() / len(y)])

In [ ]:
def linear_neuron_mse_gradients(X, y, w, b):
    """Gradients of mean((X @ w + b - y) ** 2). X: (n, features), y: (n,), w: (features,), b: (1,). Returns (dw, db)."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
exercise_rng = np.random.default_rng(11)
X_lin, y_lin = exercise_rng.normal(size=(6, 2)), exercise_rng.normal(size=6)
w_lin, b_lin = np.array([0.3, -0.7]), np.array([0.2])
linear_neuron_mse_gradients = practice("linear_neuron_mse_gradients", linear_neuron_mse_gradients,
                                       linear_neuron_mse_gradients_reference, cases=[
    (X_lin, y_lin, w_lin, b_lin),
    (X_lin[:1], y_lin[:1], w_lin, b_lin),   # a batch of one
])
dw_lin, db_lin = linear_neuron_mse_gradients(X_lin, y_lin, w_lin, b_lin)
dw_lin_num = numerical_gradient(lambda w: mse(X_lin @ w + b_lin, y_lin), w_lin)
print(f"gradient check of your formula: relative error {relative_error(dw_lin, dw_lin_num):.1e}")

### Exercise 3 · The slope of tanh

The activation $\tanh$ has the derivative $1 - \tanh^2(z)$. Write `tanh_grad(Z)` for an array `Z`, element by element. Then check it against `numerical_derivative(np.tanh, z)` at a few points of your choice, and explain why its largest value is at $z = 0$.

In [ ]:
def tanh_grad_reference(Z):
    return 1.0 - np.tanh(Z) ** 2

In [ ]:
def tanh_grad(Z):
    """Element-wise derivative of tanh: 1 - tanh(Z) ** 2."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
tanh_grad = practice("tanh_grad", tanh_grad, tanh_grad_reference, cases=[
    (np.array([-2.0, 0.0, 0.5, 3.0]),),
    (np.array([[0.1, -0.1], [10.0, -10.0]]),),
])
print("numerical slope of tanh at 0.5:", round(numerical_derivative(np.tanh, 0.5), 8))

### Exercise 4 · Open question: precision

Repeat the gradient check of section 7 in single precision: convert `X_neuron`, `w_neuron` and `b_neuron` with `.astype(np.float32)`. How large is the relative error now, and why does `eps = 1e-5` stop being a good choice? (Hint: single precision keeps about 7 significant digits instead of 16.)

## Common errors and debugging

| Symptom | Likely cause | Fix |
|---|---|---|
| the numerical gradient is all zeros or absurdly large | the array holds integers, so every nudged value is cut back to a whole number | create float arrays, for example `np.array([1.0, 2.0])` or `.astype(float)` |
| later results change after a gradient check | an entry was nudged but not restored | save the original value and assign it back |
| a correct gradient fails the check | `eps` is far too large or too small, or the array is `float32` | use `eps = 1e-5` and `float64` |
| a check fails for one or two entries only | the nudge crossed a kink, such as ReLU at 0 | check at a point away from kinks |
| `TypeError` or an array where a number is expected | `f` returns an array instead of one number | make `f` return a float, for example by taking a mean |
| errors around $10^{-6}$ look alarming or reassuring | an absolute difference was compared instead of a relative one | use `relative_error` |

The first row is the most surprising, so see it happen. Nudging the integer 1 up to 1.00001 is cut back to 1, while nudging it down to 0.99999 is cut to 0, so the "slope" comes out as an enormous 50,000. The second cell shows a kink: ReLU's slope just to the right of 0 is 1, but a central difference that straddles the kink gets it wrong.

In [ ]:
int_array = np.array([1, 2])        # integers!
int_array[0] = 1 + 1e-5             # the nudge is silently rounded away
print("integer array after a nudge:", int_array)
print("numerical gradient on integers:", numerical_gradient(sum_of_squares, np.array([1, 2])))
print("numerical gradient on floats:  ", numerical_gradient(sum_of_squares, np.array([1.0, 2.0])))

In [ ]:
for x_near_kink in [1.0, 3e-6, 0.0]:
    print(f"ReLU slope at x = {x_near_kink:g}: relu_grad gives {float(relu_grad(np.array(x_near_kink))):.2f}, "
          f"the central difference gives {numerical_derivative(relu, x_near_kink):.2f}")

At $x = 3 \times 10^{-6}$ the step of $10^{-5}$ reaches across the kink, so the estimate is $0.65$ instead of $1$. At exactly $0$ the central difference averages the two sides and returns $0.5$, while `relu_grad` uses the convention $0$. Neither is "wrong", but they disagree, so a gradient check of a ReLU network must avoid points within `eps` of a kink. Notebook 07 shows how to confirm that it did.

## Summary

- A **derivative** is a slope; a **partial derivative** is the slope along one input with the others fixed; the **gradient** collects all partial derivatives and has the shape of the parameters.
- $-\nabla L$ points downhill, so a small step against the gradient lowers the loss.
- The **central difference** has an error proportional to $h^2$. Rounding error grows like $\varepsilon / h$, so $h \approx 10^{-5}$ is the sweet spot in `float64`.
- The **chain rule** multiplies local derivatives along a path. A **backward pass** through a computational graph applies it from the loss back to every parameter. For sigmoid with binary cross-entropy, $\partial L / \partial z = p - y$.
- **Gradient checking** compares an analytic gradient with `numerical_gradient` using `relative_error`. Correct code gives about $10^{-7}$ or less, and bugs give large errors.

You built `numerical_derivative`, `numerical_gradient` and `relative_error`. The next notebook uses them under these names.

## Next

Numerical gradients are perfect for checking but far too slow for training. In **07 · Backpropagation** (`07_backpropagation.ipynb`) you derive the gradient of every weight in a network, compute all of them in one backward pass, and prove your code right with the gradient checker you just built.